# Suite YML — YAML

YAML is a thin encoding of Plain data. Loading follows the YAML 1.2 core schema; dumping quotes anything a YAML 1.1
or 1.2 reader would misread. Cases cover scalar resolution, quoting safety against a YAML 1.1 reader, line breaks
and control characters, rejection rules, and graph round trips. The `yaml` package in 1.1 mode stands in for
PyYAML's stock `safe_load`.

In [ ]:
import * as Y from "yaml";
import { readFileSync } from "node:fs";

import { Plain, Proxies, Schemas as S, YAML } from "../src/Framework/index.js";
import { DecodeError } from "../src/Framework/Errors.js";
import { assert, equal, map, raises, same_graph, text } from "./support.js";

let Item = new S.OfObject.Builder().properties(text("s"), text("i", BigInt), text("f", Number), text("b", Boolean),
  text("raw", Uint8Array)).create();
const Link = new S.OfRelation.Builder().links("from", "to").create();
Item = new S.OfObject.Builder(Item).relations((r) => r.name("out").of(Link).me("from"), (r) => r.name("in").of(Link).me("to")).update();
Proxies.register("Item", Item);
Proxies.register("Link", Link);
const B = Proxies.Builders;
const FromYAML = YAML.FromYAML(B);
const yaml11 = (t: string) => Y.parse(t, { version: "1.1", intAsBigInt: true, mapAsMap: true });
const scalar = (t: string) => (YAML.loads(`x: ${t}`) as Map<string, unknown>).get("x");

## YML-01 · Scalar resolution follows the YAML 1.2 core schema

In [ ]:
{
  const expected: [string, unknown][] = [
    ["~", null], ["null", null], ["Null", null], ["NULL", null], ["", null],
    ["true", true], ["True", true], ["TRUE", true], ["false", false], ["False", false], ["FALSE", false],
    ["yes", "yes"], ["no", "no"], ["on", "on"], ["off", "off"], ["y", "y"], ["n", "n"], ["Yes", "Yes"],
    ["0", 0n], ["-0", 0n], ["+1", 1n], ["010", 10n], ["0o17", 15n], ["0x1F", 31n], ["0x1f", 31n], ["-12", -12n],
    ["0b101", "0b101"], ["1_000", "1_000"], ["0o19", "0o19"], ["0xZZ", "0xZZ"],
    ["1.5", 1.5], [".5", 0.5], ["1.", 1.0], ["1e3", 1000.0], ["1E3", 1000.0], ["-1.5e-3", -0.0015], ["+.inf", Infinity],
    [".inf", Infinity], ["-.Inf", -Infinity], [".INF", Infinity],
    ["2026-09-01", "2026-09-01"], ["2026-09-01T09:00:00Z", "2026-09-01T09:00:00Z"], ["1:30", "1:30"],
    ["12:30:45", "12:30:45"], ["nulls", "nulls"], ["truely", "truely"], ["=", "="],
  ];
  for (const [source, value] of expected) {
    const got = scalar(source);
    assert(got === value && typeof got === typeof value, `${source} -> ${String(got)}`);
  }
  for (const source of [".nan", ".NaN", ".NAN"]) assert(Number.isNaN(scalar(source)));
}

## YML-02 · Strings that readers misread are quoted: both our loader and a YAML 1.1 reader read them back

In [ ]:
{
  const tricky = ["yes", "No", "on", "OFF", "y", "true", "null", "~", "", "010", "0o17", "0x1F", "1e3", "1_000", "1:30",
    "2026-09-01", ".inf", ".nan", "NaN", "Infinity", "-0.0", "+1", " lead", "trail ", "a: b", "- dash",
    "#hash", "x #y", '"dq"', "'sq'", "@at", "`tick", "%pct", "!bang", "&amp", "*star", "|", ">", "{}", "[]",
    ",", "?", "<<", "---", "...", "- ", ": ", "a\nb", "\t", "\x00", "\x07\x7f\x9b", "\ufeffbom", "日本語 🚀"];
  for (const value of tricky) {
    const dumped = YAML.dumps(map({ v: value }));
    assert(equal(YAML.loads(dumped), map({ v: value })), `${JSON.stringify(value)} -> ${dumped}`);
    assert(equal(yaml11(dumped), map({ v: value })), `1.1: ${JSON.stringify(value)} -> ${dumped}`);
  }
}

## YML-02b · YAML 1.1 single-letter booleans are quoted (Finding F14)
The YAML 1.1 specification makes `y`, `Y`, `n` and `N` booleans (PyYAML's own resolver omits them). The 1.1 reader
used above follows the specification, so it would misread them unquoted.

In [ ]:
for (const value of ["y", "Y", "n", "N"]) {
  assert(YAML.dumps(map({ v: value })) === `v: '${value}'\n`);
  assert(yaml11(`v: ${value}`).get("v") === (value.toLowerCase() === "y")); // the reader really does treat them as booleans
}

## YML-03 · Unicode line breaks survive (NEL, LS, PS)
Finding F1 in Python: PyYAML wrote NEL raw inside single quotes, where it folds into a space. Both implementations
double-quote such strings.

In [ ]:
for (const value of ["a\x85b", "\x85", "a\u2028b", "a\u2029b", "a\nb\x85c", "\r\n", "a\rb", "x\n", " \n y \n "]) {
  const dumped = YAML.dumps(map({ v: value }));
  assert((YAML.loads(dumped) as Map<string, unknown>).get("v") === value && (yaml11(dumped) as Map<string, unknown>).get("v") === value, dumped);
}
assert(YAML.dumps(map({ v: "a\x85b" })) === 'v: "a\\Nb"\n');

## YML-04 · Long, multi-line and whitespace-heavy strings keep their exact text

In [ ]:
for (const value of ["word ".repeat(200), "x".repeat(5000), "line\n".repeat(50), "  indented\n    more\n", "\t\ttabs\t", " ".repeat(10)]) {
  assert((YAML.loads(YAML.dumps(map({ v: value }))) as Map<string, unknown>).get("v") === value);
}

## YML-05 · Numbers keep their exact values; non-finite floats go through Plain's strings

In [ ]:
for (const value of [0n, -1n, 2n ** 64n, 2n ** 200n, -(2n ** 100n)]) {
  assert((FromYAML(Item, YAML.ToYAML(Item, B.Item().i(value).create())) as any).i === value);
}
for (const value of [0.1, 1 / 3, 1e308, 5e-324, -0.0, 1e-7, 123456789.125, 1e16, 2.0]) {
  const back = (FromYAML(Item, YAML.ToYAML(Item, B.Item().f(value).create())) as any).f;
  assert(Object.is(back, value), String(value));
}
for (const value of [Infinity, -Infinity, NaN]) {
  const out = YAML.ToYAML(Item, B.Item().f(value).create());
  assert(!out.includes(".inf") && !out.includes(".nan")); // written as the strings Plain uses
  const back = (FromYAML(Item, out) as any).f;
  assert((Number.isNaN(back) && Number.isNaN(value)) || back === value);
}

## YML-06 · Rejected input: the framework's YAML rules, with the same reason and position in every binding
One document; string keys; no duplicate keys; no tags but `!`, `!!str`, `!!seq` and `!!map`; no undefined aliases.
If there are several problems, the first in the text is reported. The table is shared verbatim with the Python
suite; rows are input and the expected message, or null when the input is accepted.

In [ ]:
{
  const RULES = globalThis.JSON.parse(String.raw`[["a: 1\na: 2", "line 2, column 1: duplicate key 'a'"], ["x:\n  a: 1\n  a: 2", "line 3, column 3: duplicate key 'a'"], ["a: {b: 1, b: 2}", "line 1, column 11: duplicate key 'b'"], ["---\na: 1\n---\nb: 2", "line 3, column 1: expected a single document, found another"], ["a: 1\n---\nb: 2", "line 2, column 1: expected a single document, found another"], ["1: x", "line 1, column 1: keys must be strings, got int 1"], ["null: 1", "line 1, column 1: keys must be strings, got NoneType None"], ["true: 1", "line 1, column 1: keys must be strings, got bool True"], ["1.5: x", "line 1, column 1: keys must be strings, got float 1.5"], ["? [a]\n: 1", "line 1, column 3: keys must be strings, got a sequence"], ["? {a: 1}\n: 1", "line 1, column 3: keys must be strings, got a mapping"], ["a:\n  2: x", "line 2, column 3: keys must be strings, got int 2"], ["!!binary AAAA", "line 1, column 1: unsupported tag !!binary"], ["a: !!binary AAAA", "line 1, column 4: unsupported tag !!binary"], ["!!set {a, b}", "line 1, column 1: unsupported tag !!set"], ["a: !!timestamp 2026-01-01", "line 1, column 4: unsupported tag !!timestamp"], ["a: !!python/object:os.system x", "line 1, column 4: unsupported tag !!python/object:os.system"], ["a: !custom x", "line 1, column 4: unsupported tag !custom"], ["a: !!int 1", "line 1, column 4: unsupported tag !!int"], ["a: !!omap []", "line 1, column 4: unsupported tag !!omap"], ["a: [*x]", "line 1, column 5: found undefined alias 'x'"], ["a: 1\na: 2\nb: [", "line 2, column 1: duplicate key 'a'"], ["a: !!int x\nb: {", "line 1, column 4: unsupported tag !!int"], ["\u00e9\ud83d\ude80: 1\n\u00e9\ud83d\ude80: 2", "line 2, column 1: duplicate key '\u00e9\ud83d\ude80'"], ["a: &x 1\nb: !!float 2", "line 2, column 4: unsupported tag !!float"], ["\u00e9\ud83d\ude80: !!int 1", "line 1, column 5: unsupported tag !!int"], ["a: ! 12", null], ["a: !!str 010", null], ["!!map {a: 1}", null], ["!!seq [1]", null], ["a: !!str\n  x", null], ["k: &a !!str v\nj: *a", null], ["a: \u0001", "line 1, column 4: unacceptable character #x0001: special characters are not allowed"], ["a: b\n# \u007f", "line 2, column 3: unacceptable character #x007f: special characters are not allowed"], ["k: \ud800", "line 1, column 4: unacceptable character #xd800: special characters are not allowed"], ["\u00e9\ud83d\ude80: \u001b", "line 1, column 5: unacceptable character #x001b: special characters are not allowed"], ["?\n", "line 1, column 2: keys must be strings, got NoneType None"], ["a: &k b\n*k : 1\n", null], ["b: 1\nc: &k b\n*k : 2\n", "line 3, column 1: duplicate key 'b'"], ["a: &x !!binary AAAA", "line 1, column 4: unsupported tag !!binary"], ["a: !!binary &x AAAA", "line 1, column 4: unsupported tag !!binary"], ["{a: 1, a: 2", "line 1, column 8: duplicate key 'a'"], ["a: 1\r\na: 2", "line 2, column 1: duplicate key 'a'"], ["&a b: 1\n&c b: 2", "line 2, column 1: duplicate key 'b'"], ["? !!str\n: 1\n? !!str\n: 2", "line 3, column 3: duplicate key ''"], ["a: 1\na: 2\n1: x", "line 2, column 1: duplicate key 'a'"], ["{1: x, a: 1, a: 2}", "line 1, column 2: keys must be strings, got int 1"], ["[*x, *y]", "line 1, column 2: found undefined alias 'x'"]]`) as [string, string | null][];
  assert(RULES.length === 48);
  for (const [source, message] of RULES) {
    try {
      YAML.loads(source);
    } catch (error) {
      assert(error instanceof DecodeError && error.message === message, `${JSON.stringify(source)}: ${String(error)}`);
      continue;
    }
    assert(message === null, source);
  }
  assert(equal(YAML.loads("a: ! 12"), map({ a: "12" }))); // the non-specific tag makes a string, as YAML 1.2 says
  assert(equal(YAML.loads("a: !!str 010"), map({ a: "010" })));
}

## YML-06b · Syntax errors: one line, with a position; the reason comes from the binding's parser

In [ ]:
for (const bad of ["a: [unclosed", "a: 'open", "\t- tab indent", "a: b: c", "a: \x01"]) {
  try {
    YAML.loads(bad);
  } catch (error) {
    assert(error instanceof DecodeError && error.line !== null && error.column !== null && !error.message.includes("\n"), `${bad}: ${String(error)}`);
    assert(error.message === `line ${error.line}, column ${error.column}: ${error.reason}`);
    continue;
  }
  throw new Error(`accepted ${JSON.stringify(bad)}`);
}
raises(DecodeError, () => YAML.loads(new Uint8Array([0x61, 0x3a, 0x20, 0xff])), "input is not valid utf-8");

## YML-07 · Accepted YAML features that map onto plain data (pinned)
Anchors and aliases to plain values are fine. The `<<` merge key is a YAML 1.1 feature, so it loads as an ordinary
key. An empty document loads as null.

In [ ]:
assert(equal(YAML.loads("a: &x 1\nb: *x"), map({ a: 1n, b: 1n })));
assert(equal((YAML.loads("base: &b {k: 1}\nd:\n  <<: *b\n  j: 2") as Map<string, unknown>).get("d"), map({ "<<": { k: 1n }, j: 2n })));
assert(YAML.loads("") === null && YAML.loads("# only a comment\n") === null);
assert(equal(YAML.loads("[1, 'two', {three: 3}]"), [1n, "two", map({ three: 3n })]));
assert(equal(YAML.loads(new TextEncoder().encode("a: 1")), map({ a: 1n })));
raises(DecodeError, () => FromYAML(Item, ""), "$: expected an object snapshot"); // an empty document is not a snapshot

## YML-08 · Dumping: no aliases, key order kept, Unicode kept

In [ ]:
{
  const shared = map({ k: 1n });
  const dumped = YAML.dumps(new Map<string, any>([["b", shared], ["a", shared], ["日", "本"]]));
  assert(!dumped.includes("&") && !dumped.includes("*"));
  assert(dumped.indexOf("b:") < dumped.indexOf("a:") && dumped.includes("日: 本"));
  // A top-level scalar: PyYAML also writes a document-end marker ("5\n...\n"); both read back the same.
  assert(YAML.dumps(5n) === "5\n" && YAML.loads(YAML.dumps(5n)) === 5n);
}

## YML-09 · Hand-written snapshots: 1.2 rules apply, and native types stay strict

In [ ]:
{
  const written = "root: s0\nobjects:\n  s0:\n    s: 2026-09-28\n    i: 010\n    f: .inf\n    b: true\n";
  const item = FromYAML(Item, written) as any;
  assert(item.s === "2026-09-28" && item.i === 10n && item.f === Infinity && item.b === true);
  raises(DecodeError, () => FromYAML(Item, "root: s0\nobjects:\n  s0:\n    b: yes\n"), "$.objects.s0.b: expected bool, got str"); // 'yes' is a string
  raises(DecodeError, () => FromYAML(Item, "root: s0\nobjects:\n  s0:\n    f: 1\n"), "$.objects.s0.f: expected float, got int"); // 1 is an int, not a float
  raises(DecodeError, () => FromYAML(Item, "root: s0\nobjects:\n  s0:\n    i: 1.0\n"), "$.objects.s0.i: expected int, got float");
}

## YML-10 · YAML agrees with Plain, and graphs round trip

In [ ]:
{
  const hub = B.Item().s("hub").raw(new Uint8Array([0x00, 0xff])).out((x: any) => x.to((y: any) => y.s("yes")))
    .out((x: any) => x.to((y: any) => y.s("010"))).create();
  const graph = Plain.ToPlain.Reachable(Item, hub);
  const text_ = YAML.ToYAML.Reachable(Item, hub);
  assert(equal(YAML.loads(text_), graph) && equal(yaml11(text_), graph));
  const back = FromYAML.Reachable(Item, text_) as Proxies.Instance;
  assert(back !== hub && same_graph(Plain.ToPlain.Reachable(Item, back), graph));
  raises(DecodeError, () => FromYAML(Item, YAML.ToYAML(Item, hub)), "$.objects.s0.out[0].to: unresolved reference 's1'");
  assert(FromYAML.OfNative(new S.OfNative.Data(String), YAML.ToYAML.OfNative(new S.OfNative.Data(String), "yes")) === "yes");
}

## YML-11 · The YAML dependency (pinned difference)
In Python, PyYAML is an optional extra imported on first use. In TypeScript, `yaml` is a regular dependency, so YAML
support is always available.

In [ ]:
{
  const pkg = globalThis.JSON.parse(readFileSync("../package.json", "utf8"));
  assert(typeof pkg.dependencies?.yaml === "string");
  assert(equal(YAML.loads("a: 1"), map({ a: 1n })));
}

## YML-13 · Every shape of plain data round-trips; only plain data can be written

In [ ]:
{
  const shapes = [
    [],
    new Map(),
    [1n, "a", null, true, 1.5, [], new Map()],
    [[1n, [2n, [3n]]], [map({ k: [{}] })], map({ a: [] })],
    map({ a: { b: { c: [1n, { d: null }] } }, e: [], f: {} }),
    map({ nan: NaN, inf: Infinity, ninf: -Infinity, big: 1e16, tiny: 5e-324, neg: -0.0 }),
    [["x"], [["y"]]],
  ];
  for (let shape of shapes as any[]) {
    const dumped = YAML.dumps(shape);
    const back: any = YAML.loads(dumped);
    const back11: any = yaml11(dumped);
    if (shape instanceof Map && shape.has("nan")) {
      assert(Number.isNaN(back.get("nan")) && Number.isNaN(back11.get("nan")));
      back.delete("nan");
      back11.delete("nan");
      shape = new Map([...shape].filter(([k]) => k !== "nan"));
      assert(Object.is(back.get("neg"), -0));
    }
    assert(equal(back, shape) && equal(back11, shape), dumped);
  }
  assert(YAML.dumps(map({ x: 1e16 })) === "x: 1.0e+16\n"); // a float keeps a '.' so every reader sees a float
  raises(TypeError, () => YAML.dumps(new Map([[1n, 2n]]) as never), "keys must be str, not int");
  raises(TypeError, () => YAML.dumps([map({ a: new Map([[1n, 2n]]) })]), "keys must be str, not int");
  for (const [value, name] of [[new Uint8Array([120]), "bytes"], [new Set([1n, 2n]), "set"], [{}, "object"]] as const) {
    raises(TypeError, () => YAML.dumps(new Map([["a", [value]]]) as never), `cannot represent ${name} as a YAML scalar`);
  }
}